<a href="https://colab.research.google.com/github/VamuveTV/3DTrajMaster/blob/main/Voice_Separation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1tH7dr2vZZEGpZHe9-wrBHFco8Ci10-_8

# Voice Separation with Pyannote.audio

This notebook uses the Pyannote.audio model to separate mixed voices in an audio file. It will identify and segment different speakers, creating a separate audio clip for each one.

In [1]:
!pip install pyannote.audio
!pip install pydub

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.7/898.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
# Please BE VERY CAREFUL, this will link your entire drive.
# So don't edit code, except the one that says 'Customize the following options',
# or you might mess up your files.
# IF YOU DO NO WANT TO LINK DRIVE, please see below for an alternative!
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [3]:
# Customize the following options!
extensions = ["mp3", "wav", "ogg", "flac"]  # We will look for all those file types.

# Paths for input and output directories on your Google Drive.
in_path = '/gdrive/MyDrive/voicesplit_input/'
out_path = '/gdrive/MyDrive/voicesplit_output/'


#Useful functions, don't forget to execute

In [6]:
import io
import os
from pathlib import Path
from pydub import AudioSegment
from pyannote.audio import Pipeline
from google.colab import files

def find_files(in_path):
    out = []
    for file in Path(in_path).iterdir():
        if file.suffix.lower().lstrip(".") in extensions:
            out.append(file)
    return out

def separate_voices(inp=None, outp=None):
    inp = inp or in_path
    outp = outp or out_path

    # Authenticate to Hugging Face Hub for the pre-trained model.
    # You need to accept the terms of use for the model.
    # Visit https://huggingface.co/pyannote/speaker-diarization and follow instructions.
    # Then run the following in a separate cell, replacing TOKEN with your actual token:
    # from huggingface_hub import notebook_login
    # notebook_login()
    # Or, paste your token directly here if you prefer.
    try:
        pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization", use_auth_token=True)
    except Exception as e:
        print(f"Error loading model. Please ensure you have authenticated with Hugging Face Hub.")
        print(e)
        return

    audio_files = find_files(inp)
    if not audio_files:
        print(f"No valid audio files in {in_path}")
        return

    print("Going to separate the following files:")
    print('\n'.join([str(f) for f in audio_files]))

    for audio_file in audio_files:
        try:
            print(f"\nProcessing {audio_file.name}...")

            # Create a specific output directory for this file
            file_output_dir = Path(outp) / audio_file.stem
            os.makedirs(file_output_dir, exist_ok=True)

            # Load the audio file
            audio = AudioSegment.from_file(audio_file)

            # Perform speaker diarization
            diarization = pipeline(audio_file)

            # Split and save the audio for each speaker
            # Fix is here: Unpack the tuple to access segment and speaker_id
            for segment, _, speaker_id in diarization.itertracks(yield_label=True):
                start_ms = segment.start * 1000
                end_ms = segment.end * 1000

                # Check for zero-length segments to avoid errors
                if start_ms < end_ms:
                    split_audio = audio[start_ms:end_ms]
                    # Save the separated audio to a file
                    output_filename = f"speaker_{speaker_id}_{int(segment.start):05d}-{int(segment.end):05d}.wav"
                    split_audio.export(file_output_dir / output_filename, format="wav")

            print(f"Separation complete for {audio_file.name}.")

        except Exception as e:
            print(f"An error occurred while processing {audio_file.name}: {e}")


#Run the separation process from Google Drive

In [7]:
separate_voices()

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/torch/pyannote/models--pyannote--segmentation/snapshots/c4c8ceafcbb3a7a280c2d357aee9fbc9b0be7f9b/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/root/.cache/torch/pyannote/sp

Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.8.0+cu126. Bad things might happen unless you revert torch to 1.x.


DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in /root/.cache/torch/pyannote/speechbrain.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/root/.cache/torch/pyannote/speechbrain/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /root/.cache/torch/pyannote/speechbrain/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/root/.cache/torch/pyannote/speechbrain/mean_var_norm_emb.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["mean_var_norm_emb"] = /root/.cache/torch/pyannote/speechbrain/mean_var_norm_emb.ckpt
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/root/.cache/torch/pyannote/speechbrain/classifier.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["classifier"] = /root/.cache/torch/pyannote/speech

Going to separate the following files:
/gdrive/MyDrive/voicesplit_input/teste.mp3

Processing teste.mp3...


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py:85: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 

Separation complete for teste.mp3.
